# Decision Tree Regression

A regression tree cuts the x-axis into pieces and predicts the average of each piece. Here it predicts **mpg** from **horsepower**.

This notebook goes with the **Decision Tree Regression** lesson on [ML Visual](https://mohammad-qt.github.io/ML_DL_Vis/). Run each cell in order with **Shift + Enter**. Nothing to install: Colab already has every library used here.

**The steps:** load the data → split → fit → compare depths → draw the staircase.

## 1. Load the data

Each row is a car from the late 1970s and early 1980s. We predict `mpg` from `horsepower`. A few cars have no horsepower listed, so we drop them first.

The data is the [Auto MPG](https://archive.ics.uci.edu/dataset/9/auto+mpg) data set from the UCI Machine Learning Repository (CC BY 4.0). See `datasets/README.md` for what we changed.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.tree import DecisionTreeRegressor, export_text

# The data lives in this project's GitHub repository, so there is nothing to upload.
DATA_URL = "https://raw.githubusercontent.com/Mohammad-QT/ML_DL_Vis/main/datasets/mpg.csv"
dataset = pd.read_csv(DATA_URL)
dataset.head()

In [ ]:
dataset = dataset.dropna(subset=['horsepower'])
X = dataset[['horsepower']].values
y = dataset['mpg'].values
print(X.shape, 'cars')

## 2. Split into train and test

The model learns from the **training** cars (75%). We hold back the other 25% and only use them at the end. The test score shows how the model does on cars it has never seen. These are the ringed points on the site's boards.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)
print(len(X_train), 'train,', len(X_test), 'test')

## 3. Fit the tree

A tree asks "is horsepower below this number?" and keeps the question that makes the two sides most alike. `max_depth` limits how many questions in a row it may ask. Trees do not need feature scaling.

In [ ]:
regressor = DecisionTreeRegressor(max_depth=3, random_state=0)
regressor.fit(X_train, y_train)
print(export_text(regressor, feature_names=['horsepower']))

## 4. Check the result

**R²** says how much of the ups and downs in the data the model explains. 1 is perfect, 0 is no better than always guessing the average. The number that matters is the **test** R². The last line shows the typical miss in miles per gallon.

In [ ]:
y_pred_train = regressor.predict(X_train)
y_pred_test = regressor.predict(X_test)
print('Train R²:', round(r2_score(y_train, y_pred_train), 3))
print('Test R²: ', round(r2_score(y_test, y_pred_test), 3))
print('Test error on average:', round(mean_absolute_error(y_test, y_pred_test), 2), 'mpg')

## 5. Choose the depth

A deep tree gets close to one step per training car, so the train R² climbs while the test R² stalls or falls. That is **overfitting**.

In [ ]:
for depth in [1, 2, 3, 4, 6, 10, None]:
    m = DecisionTreeRegressor(max_depth=depth, random_state=0).fit(X_train, y_train)
    print(f'depth {str(depth):>4}: train R² {r2_score(y_train, m.predict(X_train)):.3f}   test R² {r2_score(y_test, m.predict(X_test)):.3f}')

In [ ]:
grid = np.linspace(X.min(), X.max(), 400).reshape(-1, 1)
plt.figure(figsize=(7, 4.5))
plt.scatter(X_train, y_train, color='#5b7cfa', label='train')
plt.scatter(X_test, y_test, facecolors='white', edgecolors='#5b7cfa', linewidths=1.8, label='test')
plt.plot(grid, regressor.predict(grid), color='black', label='tree (depth 3)')
plt.xlabel('horsepower'); plt.ylabel('mpg'); plt.legend()
plt.show()

## Try this

1. Which depth gives the best test R²?
2. Print the tree at `max_depth=2` and follow it by hand for a car with 100 hp.
3. The curve is a staircase. Where would a smooth model, such as polynomial or SVR, do better?